In [2]:
import json
import glob
from datetime import datetime
from collections import defaultdict
import math

DATA_PATH = "./20251125_A5_Lagoas_MQTT/vehicle_history*.json"
# 1 sec
BIN_SIZE = 1
OUTPUT_FILE = "radar_flows.rou.xml"

def timestamp_to_seconds(ts_str):
    # 2025-11-25T10:55:32.763Z
    dt = datetime.strptime(ts_str, "%Y-%m-%dT%H:%M:%S.%fZ")
    time= dt.hour * 3600 + dt.minute * 60 + dt.second + dt.microsecond / 1e6
    return time

In [3]:
def compute_speed(path):
    if len(path) < 2:
        return 0

    first = path[0]
    last = path[-1]

    t1 = timestamp_to_seconds(first["timeOfMeasurement"])
    t2 = timestamp_to_seconds(last["timeOfMeasurement"])

    if t2 == t1:
        return 0

    x1, y1 = first["localCoordinates"]["x"], first["localCoordinates"]["y"]
    x2, y2 = last["localCoordinates"]["x"], last["localCoordinates"]["y"]

    distance = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    return distance / (t2 - t1)  # m/s

In [4]:
bins = defaultdict(lambda: {
    "count": 0,
    "width_sum": 0,
    "length_sum": 0,
    "height_sum": 0,
    "speed_sum": 0
})

vehicle_count = 0
for file_path in glob.glob(DATA_PATH):
    with open(file_path, 'r') as f:
        vehicle_count += 1
        data = json.load(f)
        path = data.get("path", [])
        
        if not path:
            continue

        path.sort(key=lambda p: timestamp_to_seconds(p["timeOfMeasurement"]))

        # dimensions
        width = sum(p["dimensions"]["width"] for p in path) / len(path)
        length = sum(p["dimensions"]["length"] for p in path) / len(path)
        height = sum(p["dimensions"]["height"] for p in path) / len(path)

        # speed
        speed = compute_speed(path)

        # better binning
        start_seconds = timestamp_to_seconds(path[0]["timeOfMeasurement"])
        bin_id = int(start_seconds // BIN_SIZE) * BIN_SIZE

        bins[bin_id]["count"] += 1
        bins[bin_id]["width_sum"] += width
        bins[bin_id]["length_sum"] += length
        bins[bin_id]["height_sum"] += height
        bins[bin_id]["speed_sum"] += speed

flows = []

for bin_id, values in bins.items():
    count = values["count"]
    if count == 0:
        continue
    flows.append({
        "timestamp": bin_id,
        "count": count,
        "avg_width": values["width_sum"] / count,
        "avg_length": values["length_sum"] / count,
        "avg_height": values["height_sum"] / count,
        "avg_speed": values["speed_sum"] / count,
    })

# sort by time
flows.sort(key=lambda x: x["timestamp"])

In [5]:
vehicle_count

847

<routes>
    <vType id="normal_car" vClass="passenger" maxSpeed="40" speedFactor="0.9" speedDev="0.2" sigma="0.5" />
    <vType id="sporty_car" vClass="passenger" maxSpeed="60" speedFactor="1.3" speedDev="0.1" sigma="0.1" />
    <vType id="trailer" vClass="trailer"  maxSpeed="30" speedFactor="1" speedDev="0.05" />
    <vType id="coach" vClass="coach"  maxSpeed="30" speedFactor="1" speedDev="0.05" />
    <flow id="normal" type="normal_car" begin="0" end="5000" number="5000" from="entry" to="exit" departSpeed="avg" departLane="best" />
    <flow id="sporty" type="sporty_car" begin="0" end="5000" number="300" from="entry" to="exit" departSpeed="avg" departLane="best" />
    <flow id="coach" type="coach" begin="0" end="5000" number="300" from="entry" to="exit" departSpeed="avg" departLane="best" />
    <flow id="trailer" type="trailer" begin="0" end="5000" number="700" from="entry" to="exit" departSpeed="avg" departLane="best" />
</routes>


In [34]:
# write a csv file with Detector;Time;qPKW;qLKW;vPKW;vLKW
with open("flows.csv", "w") as f:
    f.write("Detector;Time;qPKW;qLKW;vPKW;vLKW\n")
    for flow in flows:
        time = flow["timestamp"]
        # passenger
       
        qPKW = flow["count"] 
        vPKW = 0
        qLKW = 0
        vLKW = 0

        f.write(f"320209307_0;{time};{qPKW};{qLKW};{vPKW};{vLKW}\n")

In [35]:
# clean

import xml.etree.ElementTree as ET

input_file = "routes/edgeData.xml"
output_file = "routes/cleanEdgeData.xml"

tree = ET.parse(input_file)
root = tree.getroot()

for edge in root.iter("edge"):
    qPKW = edge.get("qPKW")

    for attr in ["qLKW", "qPKW", "groups"]:
        if attr in edge.attrib:
            del edge.attrib[attr]

    if qPKW is not None:
        edge.set("entered", qPKW)

tree.write(output_file, encoding="utf-8", xml_declaration=True)

In [9]:
# add vehicle types

import xml.etree.ElementTree as ET

input_file = "routes/sampledRoutes.rou.xml"
output_file = "routes/vehicleRoutes.rou.xml"

tree = ET.parse(input_file)
root = tree.getroot()

vdist = ET.Element("vTypeDistribution", {"id": "typedist1"})

vtypes = [
    ("cautious", {
        "accel": "0.8", "decel": "4.0", "sigma": "0.3",
        "tau": "1.5", "maxSpeed": "25",
        "speedFactor": "0.9", "probability": "0.3"
    }),
    ("normal", {
        "accel": "1.2", "decel": "4.5", "sigma": "0.5",
        "tau": "1.0", "maxSpeed": "30",
        "speedFactor": "1.0", "probability": "0.5"
    }),
    ("aggressive", {
        "accel": "2.0", "decel": "5.0", "sigma": "0.8",
        "tau": "0.6", "maxSpeed": "35",
        "speedFactor": "1.2", "probability": "0.2"
    })
]

for vid, attrs in vtypes:
    v = ET.Element("vType", {"id": vid, **attrs})
    vdist.append(v)

root.insert(0, vdist)

for vehicle in root.iter("vehicle"):
    vehicle.set("type", "typedist1")
    vehicle.set("departSpeed", "random")

tree.write(output_file, encoding="utf-8", xml_declaration=True)